# Stand-alone and OpenFL federated learning (simulated with workflow interface)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Prepare environment

In [ ]:
!ls -l

In [ ]:
# we will use pytorch_geometric and OpenFL in this notebook
!pip install git+https://github.com/securefederatedai/openfl.git
!pip install -r workflow_interface_requirements.txt

In [ ]:
# Uncomment this if running in Google Colab
!pip install -r https://raw.githubusercontent.com/intel/openfl/develop/openfl-tutorials/experimental/workflow/workflow_interface_requirements.txt
import os
os.environ["USERNAME"] = "colab"

In [ ]:
# to fix a current issue:
!curl https://raw.githubusercontent.com/protocolbuffers/protobuf/main/python/google/protobuf/internal/builder.py > /usr/local/lib/python3.10/dist-packages/google/protobuf/internal/builder.py


In [ ]:
!pip install torch_geometric

## Stand-Alone

In [ ]:
import argparse
import os.path as osp

import torch
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.logging import init_wandb, log
from torch_geometric.nn import MLP, GINConv, global_add_pool

In [ ]:
# parser = argparse.ArgumentParser()
# parser.add_argument('--dataset', type=str, default='MUTAG')
# parser.add_argument('--batch_size', type=int, default=128)
# parser.add_argument('--hidden_channels', type=int, default=32)
# parser.add_argument('--num_layers', type=int, default=5)
# parser.add_argument('--lr', type=float, default=0.01)
# parser.add_argument('--epochs', type=int, default=100)
# parser.add_argument('--wandb', action='store_true', help='Track experiment')
# args = parser.parse_args()

class dummy:
    pass
args = dummy()

args.dataset='MUTAG'
args.batch_size=32
args.hidden_channels=32
args.num_layers=5
args.lr=0.01
args.momentum=0.9
args.epochs=100
args.wandb=False

In [ ]:
## tensorboard unique id for runs
import random
unique_id_stand_alone = random.randint(10,2000000) # random 'unique' id for tensorboard clarity
print("Last Run ID is: ", unique_id_stand_alone)

In [ ]:
# tensorboard
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('./logs/pyg_and_openFL_standalone_reusing_PYG_Example{}_BS{}'.format(unique_id_stand_alone, args.batch_size), flush_secs=5)

In [ ]:
def write_metric(node_name, task_name, metric_name, metric, round_number):
    writer.add_scalar("{}/{}/{}".format(node_name, task_name, metric_name),
                      metric, round_number)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    # MPS is currently slower than CPU due to missing int64 min/max ops
    device = torch.device('cpu')
else:
    device = torch.device('cpu')

print(device)

In [ ]:
init_wandb(name=f'GIN-{args.dataset}', batch_size=args.batch_size, lr=args.lr,
           epochs=args.epochs, hidden_channels=args.hidden_channels,
           num_layers=args.num_layers, device=device)

path = osp.join(osp.dirname(osp.realpath( "__file__")), '..', 'data', 'TU')
dataset = TUDataset(path, name=args.dataset).shuffle()

train_dataset = dataset[len(dataset) // 10:]
train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True)

test_dataset = dataset[:len(dataset) // 10]
test_loader = DataLoader(test_dataset, args.batch_size)

In [ ]:
dataset.num_classes

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super().__init__()

        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            mlp = MLP([in_channels, hidden_channels, hidden_channels])
            self.convs.append(GINConv(nn=mlp, train_eps=False))
            in_channels = hidden_channels

        self.mlp = MLP([hidden_channels, hidden_channels, out_channels],
                       norm=None, dropout=0.5)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = conv(x, edge_index).relu()
        x = global_add_pool(x, batch)
        return self.mlp(x)

In [ ]:
model = Net(dataset.num_features, args.hidden_channels, dataset.num_classes,
            args.num_layers).to(device)
#optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
optimizer = torch.optim.SGD(model.parameters(), lr=args.lr,momentum=args.momentum)

In [ ]:
def train(model, loader):
    model.train()

    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * data.num_graphs
    return total_loss / len(train_loader.dataset)

In [ ]:
@torch.no_grad()
def test(model, loader):
    model.eval()

    total_correct = 0
    for data in loader:
        data = data.to(device)
        pred = model(data.x, data.edge_index, data.batch).argmax(dim=-1)
        total_correct += int((pred == data.y).sum())
        accuracy = total_correct / len(loader.dataset)
        # print("Accuracy on the passed data is: ", accuracy)
    return accuracy


In [ ]:
identity = "Original"
for epoch in range(1, args.epochs + 1):
    loss = train(model, train_loader)
    train_acc = test(model, train_loader)
    test_acc = test(model, test_loader)
    write_metric(identity, "train", "loss", loss, epoch)
    write_metric(identity, "train", "accuracy", train_acc, epoch)
    write_metric(identity, "test", "accuracy", test_acc, epoch)
    log(Epoch=epoch, Loss=loss, Train=train_acc, Test=test_acc)

## OpenFL

In [ ]:
log_interval = 100
BS = args.batch_size
unique_id_openfl = random.randint(10,200000)
# tensorboard
# from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('./logs/pyg_and_openFL_reusing_PYG_Example_OPENFL_{}_BS{}'.format(unique_id_openfl, BS), flush_secs=5)
print(unique_id_openfl)

In [ ]:
#openFL imports
from openfl.experimental.workflow.interface import FLSpec, Aggregator, Collaborator
from openfl.experimental.workflow.runtime import LocalRuntime
from openfl.experimental.workflow.placement import aggregator, collaborator

In [ ]:
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')
from copy import deepcopy

In [ ]:
# OpenFL: Averaging all parameter vectors
def FedAvg(models): 
    new_model = models[0]
    state_dict_keys = new_model.state_dict().keys()
    model_dict_list = [mdl.state_dict() for mdl in models]
    state_dict = new_model.state_dict()
    num_models = len(models)
    for key in state_dict_keys:
        state_dict[key] = torch.sum(
            torch.stack(
            [mdldict[key]/num_models for mdldict in model_dict_list]), dim=0)
    new_model.load_state_dict(state_dict)
    return new_model

In [ ]:
#OpenFL: definition of compute flow
class FederatedFlow(FLSpec):
    def __init__(self, model = None, optimizer = None, rounds=3, **kwargs):
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.optimizer = optimizer #optimizer(params=self.model.parameters()) #
        else:
            raise Exception("Sorry, model must be defined")
        self.rounds = rounds
        self.aggr_training_value_list = []

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.private = 10
        self.current_round = 0
        self.next(self.aggregated_model_validation,foreach='collaborators',exclude=['private'])

    @collaborator
    def aggregated_model_validation(self):
        print(f'########################## Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = test(self.model, self.test_loader)
        self.next(self.train)

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=args.lr)
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=args.lr, momentum=args.momentum)
        # NOTE: it seems that we need to define here the opt fot it to be able to act on collaborator parameters
        total_loss = 0
        # print("Collaborator self.model params example BEFORE a round of training:\n ", self.model.state_dict()["lin0.weight"])
        ll = len(self.train_loader)
        for batch_idx, data in enumerate(self.train_loader):
            data = data.to(device)
            self.optimizer.zero_grad()
            out = self.model(data.x, data.edge_index, data.batch)
            loss = F.cross_entropy(out, data.y)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * data.num_graphs
            self.loss = loss.item()
        # print("Collaborator self.model params example AFTER a round of training:\n ", self.model.state_dict()["lin0.weight"])
        write_metric(self.input, "train", "loss", loss.item(), self.current_round)
        self.training_completed = True
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = test(self.model, self.test_loader)
        write_metric(self.input, "test", "accuracy", self.local_validation_score, self.current_round)
        self.next(self.join, exclude=['training_completed'])

    @aggregator
    def join(self, inputs):
        print(f'############## AGGREGATOR NOW: time to join!! for Epoch {self.current_round}')
        self.average_loss = sum(input.loss for input in inputs)/len(inputs)
        self.aggr_training_value_list.append(self.average_loss)
        self.aggregated_model_accuracy = sum(input.agg_validation_score for input in inputs)/len(inputs)
        self.local_model_accuracy = sum(input.local_validation_score for input in inputs)/len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')

        self.model = FedAvg([input.model for input in inputs])
        self.optimizer = [input.optimizer for input in inputs][0]

        write_metric("Aggregator", "train", "avg_loss", self.average_loss, self.current_round)
        write_metric("Aggregator", "test", "accuracy", self.aggregated_model_accuracy, self.current_round)
        print("Wrote TensorBoard information for Aggregator ")

        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation, foreach='collaborators', exclude=['private'])
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print("List of averaged training losses :\n", self.aggr_training_value_list)
        print(f'This is the end of the flow')

In [ ]:
# for the data we have already these object from the stand alone part:
# train_dataset = dataset[len(dataset) // 10:]
# train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True)

# test_dataset = dataset[:len(dataset) // 10]
# test_loader = DataLoader(test_dataset, args.batch_size)

#also we can reuse the model definition, instantiating a new object (model) that will be trained
model_OpenFL = Net(dataset.num_features, args.hidden_channels, dataset.num_classes,
            args.num_layers)
model = model_OpenFL.to(device)

#optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
optimizer = torch.optim.SGD(model.parameters(), lr=args.lr,momentum=args.momentum)

In [ ]:
# Setup participants
aggregator = Aggregator()
aggregator.private_attributes = {}

# Setup collaborators with private attributes
collaborator_names = ['Coll_1', 'Coll_2']
collaborators = [Collaborator(name=name) for name in collaborator_names]

for idx, collaborator in enumerate(collaborators):
    local_train = deepcopy(train_dataset)
    local_test = deepcopy(test_dataset)
    test_l = len(local_test)
    train_l = len(local_train)
    local_train = train_dataset[idx*train_l//2: (idx+1)*train_l//2] # this is ok only for 2 collaborators....
    local_test = test_dataset[idx*test_l//2: (idx+1)*test_l//2]
    collaborator.private_attributes = {
            'train_loader': DataLoader(local_train, batch_size=BS, shuffle=True),
            'test_loader': DataLoader(local_test, batch_size=BS)
    }

local_runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators, backend='single_process')
print(f'Local runtime collaborators = {local_runtime.collaborators}')

best_model = None
optimizer = None
flflow = FederatedFlow(model, optimizer, rounds=args.epochs)
flflow.runtime = local_runtime

In [ ]:
print("NOW CALLING .run -----------------------------------------------------------------------------------------------------")
flflow.run()
print(".run has finished-----------------------------------------------------------------------------------------------------")

print(f'\nFinal aggregated model accuracy for {flflow.rounds} rounds of training: {flflow.aggregated_model_accuracy}')

